In [1]:
import json
import pandas as pd
import plotly.express as px

with open("traffic.jsonl", encoding="utf-8") as file:
    df = pd.DataFrame(json.loads(line) for line in file if line.strip())

# Feltet er stavet "depature" i filen.
df["departure"] = pd.to_datetime(df["depature"], format="%H:%M")
df["arrival"] = pd.to_datetime(df["arrival"], format="%H:%M")

df["duration"] = (
    (df["arrival"] - df["departure"]).dt.total_seconds() / 60
)

for route, data in df.groupby("road"):
    fig = px.scatter(
        data,
        x="departure",
        y="duration",
        title=f"Rute: {route}",
        labels={
            "departure": "Avgangstid",
            "duration": "Reisetid (minutter)"
        }
    )

    fig.update_xaxes(tickformat="%H:%M")
    fig.update_traces(
        hovertemplate="Avgang: %{x|%H:%M}<br>Reisetid: %{y} min<extra></extra>"
    )
    fig.show(renderer="notebook_connected")

In [2]:
fig = px.scatter(
    df,
    x="departure",
    y="duration",
    color="road",
    title="Reisetid for alle ruter",
    labels={
        "departure": "Avgangstid",
        "duration": "Reisetid (minutter)",
        "road": "Rute"
    },
    opacity=0.6
)

fig.update_xaxes(tickformat="%H:%M")
fig.update_traces(
    hovertemplate="Avgang: %{x|%H:%M}<br>Reisetid: %{y} min"
)
fig.show(renderer="notebook_connected")

In [3]:
import json
import pandas as pd
import plotly.express as px

with open("traffic.jsonl", encoding="utf-8") as file:
    df = pd.DataFrame(
        json.loads(line) for line in file if line.strip()
    )


def to_minutes(time):
    hours, minutes = map(int, time.split(":"))
    return hours * 60 + minutes


# Konverter begge klokkeslettene til minutter siden midnatt.
df["departure_min"] = df["depature"].map(to_minutes)
df["arrival_min"] = df["arrival"].map(to_minutes)

# Reisetid i minutter.
df["duration"] = df["arrival_min"] - df["departure_min"]

# Z-score på departure.
departure_mean = df["departure_min"].mean()
departure_std = df["departure_min"].std(ddof=0)

df["departure_z"] = (
    (df["departure_min"] - departure_mean) / departure_std
)

# Kontroller reisetiden for hver rute.
print(df.groupby("road")["duration"].agg(["min", "max"]))

fig = px.scatter(
    df,
    x="departure_min",
    y="duration",
    color="road",
    title="Reisetid for alle ruter",
    labels={
        "departure_min": "Avgangstid",
        "duration": "Reisetid (minutter)",
        "road": "Rute"
    },
    custom_data=["depature"],
    opacity=0.6
)

hours = range(7, 18)
fig.update_xaxes(
    tickvals=[h * 60 for h in hours],
    ticktext=[f"{h:02d}:00" for h in hours]
)

fig.update_traces(
    hovertemplate="Avgang: %{customdata[0]}<br>Reisetid: %{y} min"
)

fig.show(renderer="notebook_connected")

         min  max
road             
A->C->D   69  133
A->C->E   86  109
B->C->D   50  163
B->C->E   66  137


In [4]:
import json
import pandas as pd
import plotly.express as px

with open("traffic.jsonl", encoding="utf-8") as file:
    df = pd.DataFrame(
        json.loads(line) for line in file if line.strip()
    )

# Feltet er stavet "depature" i filen.
df["departure"] = pd.to_datetime(df["depature"], format="%H:%M")
df["arrival"] = pd.to_datetime(df["arrival"], format="%H:%M")

df["duration"] = (
    (df["arrival"] - df["departure"]).dt.total_seconds() / 60
)

# Avgangstid i minutter siden midnatt.
df["departure_min"] = (
    df["departure"].dt.hour * 60
    + df["departure"].dt.minute
)

# Felles z-score-skalering for alle rutene.
departure_mean = df["departure_min"].mean()
departure_std = df["departure_min"].std(ddof=0)

df["departure_z"] = (
    (df["departure_min"] - departure_mean) / departure_std
)

for route, data in df.groupby("road"):
    fig = px.scatter(
        data,
        x="departure_z",
        y="duration",
        title=f"Rute: {route}",
        labels={
            "departure_z": "Avgangstid (z-score)",
            "duration": "Reisetid (minutter)"
        },
        custom_data=["depature"]
    )

    fig.update_traces(
        hovertemplate=(
            "Avgang: %{customdata[0]}<br>"
            "Z-score: %{x:.2f}<br>"
            "Reisetid: %{y} min<extra></extra>"
        )
    )

    fig.show(renderer="notebook_connected")